In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dt = pd.read_csv('../data/processed/train_data.csv')

In [2]:
#Identify numeric columns
numeric_cols = dt.select_dtypes(include=['int64', 'float64']).columns

# Find potential categorical columns based on unique value count
potential_categorical = []
threshold = 100 

for col in numeric_cols:
    unique_count = dt[col].nunique()
    
    # Exclude the target variable from this list to prevent data leakage
    if unique_count < threshold and col != 'isFraud':
        potential_categorical.append(col)

print(f"Found {len(potential_categorical)} disguised categorical columns.")

#Convert them to 'category' data type
dt[potential_categorical] = dt[potential_categorical].astype('category')

print("Type casting completed. Memory usage optimized.")

Found 248 disguised categorical columns.
Type casting completed. Memory usage optimized.


In [3]:
v_138_166 = [f'V{i}' for i in range(138, 167)]
v_322_339 = [f'V{i}' for i in range(322, 340)]
id_series = [f'id_{str(i).zfill(2)}' for i in [24, 25, 7, 8, 21, 26, 27, 23, 22, 18, 4, 3, 33, 9, 10, 30, 32, 34, 14]]

other_cols = ['dist2', 'D7', 'D13', 'D14', 'D12', 'D6', 'D8', 'D9','addr2', 'DeviceInfo', 'TransactionID', 'TransactionDT']

high_null_cols = v_138_166 + v_322_339 + id_series + other_cols

dt1 = dt.drop(columns=high_null_cols)

In [4]:
dt1.head(5)

,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,id_19,id_20,id_28,id_29,id_31,id_35,id_36,id_37,id_38,DeviceType
0,0,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,...,542.0,144.0,New,NotFound,samsung browser 6.2,T,F,T,T,mobile


In [5]:
# --- MISSING VALUE ANALYSIS ---
print("--- MISSING VALUE ANALYSIS ---")

# Calculate missing value percentages for all columns
missing_values = dt1.isnull().sum()
missing_percentages = (missing_values / len(dt1)) * 100

# Create a DataFrame for better viewing
missing_df = pd.DataFrame({'Missing_Count': missing_values, 'Missing_Percentage': missing_percentages})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} out of {dt.shape[1]}")
print("\nTop 15 columns with most missing values:")
display(missing_df.head(15))

--- MISSING VALUE ANALYSIS ---
Total columns with missing values: 338 out of 434

Top 15 columns with most missing values:


,Missing_Count,Missing_Percentage
id_13,463220,78.440072
id_16,461200,78.098012
V276,460110,77.913435
V277,460110,77.913435
V278,460110,77.913435
V273,460110,77.913435
V269,460110,77.913435
V268,460110,77.913435
V267,460110,77.913435
V266,460110,77.913435


In [6]:
import numpy as np

def auto_impute_missing_values(df):
    """
    Automated missing value imputation with updated thresholds:
    - Ratio < 5%: Fill with Median/Mode.
    - Ratio 5% - 30%: Fill with Median/Mode (Fast alternative to KNN).
    - Ratio 30% - 80%: Create 'is_missing' flag and fill with placeholder.
    - Ratio > 80%: Skip.
    """
    total_rows = len(df)
    
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_ratio = missing_count / total_rows
        
        # 1. Skip if no missing values or ratio > 80%
        if missing_ratio == 0 or missing_ratio > 0.80:
            continue
            
        # 2. Ratio 30% - 80%: Indicator Flagging
        elif missing_ratio >= 0.30:
            df[f'{col}_is_missing'] = df[col].isnull().astype(np.int8)
            
            if df[col].dtype.name == 'category':
                df[col] = df[col].cat.add_categories('Missing_Flagged')
                df[col] = df[col].fillna('Missing_Flagged')
            elif df[col].dtype == 'object':
                df[col] = df[col].fillna('Missing_Flagged')
            else:
                df[col] = df[col].fillna(-999)
                
        # 3. Ratio 5% - 30%: Fast Imputation (Placeholder for Model-based)
        elif missing_ratio >= 0.05:
            if df[col].dtype.name == 'category' or df[col].dtype == 'object':
                mode_val = df[col].mode()[0]
                df[col] = df[col].fillna(mode_val)
            else:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                
        # 4. Ratio < 5%: Simple Imputation
        else:
            if df[col].dtype.name == 'category' or df[col].dtype == 'object':
                mode_val = df[col].mode()[0]
                df[col] = df[col].fillna(mode_val)
            else:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                
    return df

# Apply the updated function
dt1 = auto_impute_missing_values(dt1)
print(f"Imputation completed with new thresholds. New dataset shape: {dt1.shape}")

C:\Users\Kruger\AppData\Local\Temp\ipykernel_19468\493544840.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_is_missing'] = df[col].isnull().astype(np.int8)
C:\Users\Kruger\AppData\Local\Temp\ipykernel_19468\493544840.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_is_missing'] = df[col].isnull().astype(np.int8)
C:\Users\Kruger\AppData\Local\Temp\ipykernel_19468\493544840.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

Imputation completed with new thresholds. New dataset shape: (590540, 513)


C:\Users\Kruger\AppData\Local\Temp\ipykernel_19468\493544840.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_is_missing'] = df[col].isnull().astype(np.int8)


In [7]:
target_mean = dt1.groupby('P_emaildomain')['isFraud'].mean()
dt1['P_emaildomain'] = dt1['P_emaildomain'].map(target_mean).fillna(0)

dt1 = pd.get_dummies(dt1, columns=['card4', 'card6'], drop_first=True, dtype=int)

object_cols = dt1.select_dtypes(include=['object']).columns
dt1[object_cols] = dt1[object_cols].astype('category')


In [8]:
dt1.head()

,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card5,addr1,dist1,P_emaildomain,...,id_36_is_missing,id_37_is_missing,id_38_is_missing,DeviceType_is_missing,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0,68.5,W,13926,361.0,150.0,142.0,315.0,19.0,0.039444,...,1,1,1,1,1,0,0,1,0,0
1,0,29.0,W,2755,404.0,150.0,102.0,325.0,-999.0,0.039444,...,1,1,1,1,0,1,0,1,0,0
2,0,59.0,W,4663,490.0,150.0,166.0,330.0,287.0,0.094584,...,1,1,1,1,0,0,1,0,1,0
3,0,50.0,W,18132,567.0,150.0,117.0,476.0,-999.0,0.022757,...,1,1,1,1,0,1,0,0,1,0
4,0,50.0,H,4497,514.0,150.0,102.0,420.0,-999.0,0.039444,...,0,0,0,0,0,1,0,1,0,0


In [9]:
dt1.to_csv('../data/processed/train_cleaned.csv', index=False)